# A1 · Model exploration

**Status.** Exploration. Nothing here is a paper artifact or one of the paper's supporting checks. These are looks at the forecasts that informed decisions while the analysis was being built, kept so that they stay reproducible. The decisions they informed are recorded in `publication/01_conference/analysis_plan.md`.

**Purpose.** Reproduce three explorations:

- a first look at each model's POE50 against the actuals, made before any score was computed;
- the day M3 predicts as lowest in each test year, which led to the single non-working-day flag (F3, D19);
- the miss on public holidays, holiday by holiday, which showed where the holiday effect left by that flag sits (F4, D21).

**Inputs**

- `publication/01_conference/data/interim/forecasts.parquet` from notebook 05.
- `publication/01_conference/data/interim/network_30min.parquet` from notebook 01.
- `publication/01_conference/data/interim/pv_daily.parquet` and `weather_daily.parquet` from notebook 02.

**Outputs** (`exploration/01_conference/outputs`)

- `first_look_poe50_daytime_low.csv` and `first_look_poe50_minimum.csv`.
- `check_lowest_predicted_day_m3.csv`.
- `holiday_miss_by_holiday.csv` and `holiday_miss_christmas_run.csv`.

**Main process**

1. Set up.
2. First look: POE50 against the actuals.
3. The day M3 predicts as lowest.
4. The holiday miss, holiday by holiday.
5. Result.

Run notebooks 01, 02 and 05 first.

## 1. Set up

In [1]:
import numpy as np
import pandas as pd

from pynmdf import evaluate, targets
from pynmdf.config import load_config

cfg = load_config()
interim, archive = cfg["dirs"]["interim"], cfg["dirs"]["exploration"]
subdivision = cfg["calendar"]["holiday_subdivision"]

pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_rows", 60)
pd.set_option("display.float_format", "{:,.0f}".format)
forecasts = pd.read_parquet(interim / "forecasts.parquet")
total = pd.read_parquet(interim / "network_30min.parquet")["total_mw"]
pv = pd.read_parquet(interim / "pv_daily.parquet")["pv_mw"]
weather = pd.read_parquet(interim / "weather_daily.parquet")

## 2. First look: POE50 against the actuals

Annual forecasts only, one table per target: the actual value and each model's POE50, by test year. This was the first look at the forecasts, before any score; the scores are in notebook 06.

In [2]:
def annual_view(target: str) -> pd.DataFrame:
    """Actual and each model's POE50 for one annual target, by test year."""
    rows = forecasts[(forecasts["target"] == target) & (forecasts["level"] == "annual")]
    view = rows.pivot(index="year", columns="model", values="poe50")
    view.insert(0, "actual", rows.groupby("year")["actual"].first())
    view.to_csv(archive / f"first_look_poe50_{target}.csv")
    return view


print("Daytime low (MW)")
annual_view("daytime_low")

Daytime low (MW)


model,actual,M1,M2,M3,M4,M5,M6
year,,,,,,,
2019,"1,759","1,915","1,789","1,782","1,721","1,810","1,816"
2020,"1,671","1,764","1,738","1,639","1,576","1,680","1,674"
2021,"1,357","1,668","1,584","1,426","1,307","1,410","1,443"
2022,"1,458","1,312","1,323","1,110",999,"1,293","1,344"
2023,"1,080","1,455","1,251","1,016",921,"1,332","1,305"
2024,"1,019","1,034","1,049",875,757,994,"1,129"
2025,880,958,925,839,683,881,858


In [3]:
print("Whole-day minimum (MW)")
annual_view("minimum")

Whole-day minimum (MW)


model,actual,M1,M2,M3,M4,M5,M6
year,,,,,,,
2019,"1,473","1,537","1,544","1,500","1,501","1,436","1,516"
2020,"1,429","1,504","1,538","1,525","1,507","1,422","1,428"
2021,"1,357","1,450","1,492","1,426","1,307","1,338","1,382"
2022,"1,428","1,357","1,323","1,110",999,"1,272","1,339"
2023,"1,080","1,449","1,251","1,016",921,"1,321","1,305"
2024,"1,019","1,079","1,049",875,757,994,"1,129"
2025,880,997,925,839,683,881,858


## 3. The day M3 predicts as lowest

The annual low is set by a model's most extreme day, so that is where a model's weak points show. Each model is fitted up to the origin and then given the target year's **own** weather, with no replay and no borrowed miss. That is not a forecast (the weather is not known a year ahead); it shows what the fitted model says about each day once its inputs are known.

The table is for M3 and the daytime low. For each test year it gives the day M3 predicts as lowest, whether it is a public holiday, the prediction and what that day actually did; then the day that was actually lowest, its value and what M3 predicted for it.

With separate weekend and public-holiday flags, the lowest predicted day was a holiday falling on a weekend in six of seven years, predicted far below what such days do. That is what led to the single non-working-day flag; this table shows the models as they are now, with that flag.

In [4]:
daily = evaluate.own_weather_predictions(total, pv, weather, cfg)

lowest_day = evaluate.lowest_predicted_days(daily, "M3", "daytime", subdivision)
lowest_day.to_csv(archive / "check_lowest_predicted_day_m3.csv")
lowest_day

,lowest_predicted_day,public_holiday,predicted_mw,actual_that_day_mw,lowest_actual_day,actual_low_mw,predicted_for_it_mw
year,,,,,,,
2019,Sun 01 Sep,,"2,065","1,928",Sun 20 Oct,"1,759","2,080"
2020,Sun 27 Sep,,"1,857","1,750",Fri 25 Dec,"1,671","2,068"
2021,Sun 26 Sep,,"1,605","1,866",Sun 17 Oct,"1,357","1,655"
2022,Sun 25 Sep,,"1,295","1,458",Sun 25 Sep,"1,458","1,295"
2023,Sun 31 Dec,,"1,294","1,840",Sun 29 Oct,"1,080","1,374"
2024,Sun 27 Oct,,"1,111","1,021",Sat 26 Oct,"1,019","1,111"
2025,Sun 26 Oct,,"1,046","1,456",Sat 27 Dec,880,"1,350"


## 4. The holiday miss, holiday by holiday

The supporting check in notebook 05 shows that the models miss public holidays on the low side by more than they miss ordinary weekend days. This section splits that miss, to see where it comes from. A miss is actual minus predicted, in MW, from the same own-weather predictions as section 3.

The first table gives the mean miss on each public holiday, by model and for each low, with the number of holiday days over the seven test years. The column `M2 to M5` is the mean over the four models that existed when this look was first made.

In [5]:
scored = daily.dropna(subset=["actual"]).copy()
scored["miss"] = scored["actual"] - scored["predicted"]
days = pd.DatetimeIndex(scored["date"].unique())
holiday_name = pd.Series(targets.public_holidays(days, subdivision).to_numpy(), index=days)
scored["holiday"] = scored["date"].map(holiday_name)

held = scored[scored["holiday"] != ""]
by_holiday = held.groupby(["low", "holiday", "model"])["miss"].mean().unstack("model")
by_holiday["M2 to M5"] = by_holiday[["M2", "M3", "M4", "M5"]].mean(axis=1)
by_holiday["days"] = held[held["model"] == "M2"].groupby(["low", "holiday"]).size()
by_holiday.to_csv(archive / "holiday_miss_by_holiday.csv")
by_holiday

model                                                       M2   M3   M4   M5   M6  M2 to M5  days
low       holiday                                                                                 
daytime   ANZAC Day                                       -174  -70  -31  -58  -74       -83     6
          Australia Day                                      2   90   79  120  225        73     6
          Boxing Day                                       -74   65   -5 -175  -37       -47     7
          Boxing Day (observed)                           -243  -98 -142  -83 -102      -141     2
          Christmas Day                                   -271 -147 -128 -167 -197      -178     6
          Christmas Day (observed)                        -108   78  122   -6  -32        22     2
          Easter Monday                                   -207 -105  -65 -185 -106      -140     6
          Easter Saturday                                 -333 -234 -146 -140 -206      -213     6
          Easter Sunday                                   -434 -334 -226 -228 -307      -305     6
          Good Friday                                     -277 -178 -146 -137 -150      -184     6
          King's Birthday                                  -19   93  107   29  239        52     3
          Labour Day                                       -98   22  -36 -175 -253       -72     7
          National Day of Mourning for Queen Elizabeth II  582  787  545  310  453       556     1
          New Year's Day                                  -533 -446 -375 -228 -310      -396     6
          New Year's Day (observed)                       -371 -179  -21 -121 -145      -173     2
          Queen's Birthday                                -126  -96  -96   53  -38       -66     4
overnight ANZAC Day                                        -65  -70  -64   -9  -23       -52     6
          Australia Day                                      8   -7  -13   45   56         8     6
          Boxing Day                                       -87  -69  -75  -54  -38       -71     7
          Boxing Day (observed)                           -128 -101 -104 -122  -44      -114     2
          Christmas Day                                    -53  -37  -36   19    8       -27     6
          Christmas Day (observed)                         -49  -10   -2   33   27        -7     2
          Easter Monday                                    -85  -90  -86  -66  -44       -82     6
          Easter Saturday                                 -127 -134 -131  -53  -81      -111     6
          Easter Sunday                                   -129 -136 -125  -63  -83      -113     6
          Good Friday                                      -87  -94  -91  -30  -41       -75     6
          King's Birthday                                  -70  -76  -74  -23   66       -61     3
          Labour Day                                       -63  -57  -69  -35  -68       -56     7
          National Day of Mourning for Queen Elizabeth II   65  113  100  147  125       106     1
          New Year's Day                                  -181 -194 -180  -62 -136      -154     6
          New Year's Day (observed)                       -176 -142 -122  -93 -192      -133     2
          Queen's Birthday                                 -69  -88  -87   35   -1       -52     4

The second table separates the days from 24 December to 2 January from the rest of the year, for public holidays, ordinary weekend days and working days alike. If working days in that run are missed as badly as its holidays, the miss there belongs to the Christmas shutdown and not to the holiday flag.

In [6]:
month_day = scored["date"].dt.strftime("%m-%d")
in_run = (month_day >= "12-24") | (month_day <= "01-02")
is_holiday = scored["holiday"] != ""
is_weekend = scored["date"].dt.dayofweek >= 5
by_kind = scored.assign(
    period=np.where(in_run, "24 Dec to 2 Jan", "rest of year"),
    kind=np.select([is_holiday, is_weekend], ["holiday", "weekend"], default="working"),
)
keys = ["low", "period", "kind"]
christmas_run = by_kind.groupby([*keys, "model"])["miss"].mean().unstack("model")
christmas_run["days"] = by_kind[by_kind["model"] == "M2"].groupby(keys).size()
christmas_run.to_csv(archive / "holiday_miss_christmas_run.csv")
christmas_run

model                               M2   M3   M4   M5   M6  days
low       period          kind                                  
daytime   24 Dec to 2 Jan holiday -267 -138 -127 -161 -158    24
                          weekend -239 -102  -88 -174 -221    14
                          working -426 -302 -307 -281 -309    29
          rest of year    holiday -184  -84  -59  -88  -87    52
                          weekend  -47   53   45   20   -9   653
                          working -131  -29  -35   52    1  1642
overnight 24 Dec to 2 Jan holiday -104  -91  -87  -34  -50    24
                          weekend -107  -93  -90  -44  -79    14
                          working -109  -97 -101  -50  -65    29
          rest of year    holiday  -77  -82  -81  -24  -33    52
                          weekend  -39  -39  -40   23    6   653
                          working  -44  -43  -43   20   12  1642

## 5. Result

Where the holiday effect on the daytime low sits, in two lines of numbers: the holidays with the largest mean miss, and the mean miss of M3 on each kind of day inside and outside the Christmas to New Year run.

In [7]:
largest = by_holiday.loc["daytime", "M2 to M5"].sort_values().head(6)
print("Daytime low: the six public holidays with the largest mean miss, M2 to M5 (MW)")
print(largest.round().to_string())
print()
print("Daytime low: mean miss of M3 by period and kind of day (MW)")
print(christmas_run.loc["daytime", "M3"].unstack("kind").round().to_string())

Daytime low: the six public holidays with the largest mean miss, M2 to M5 (MW)
holiday
New Year's Day              -396
Easter Sunday               -305
Easter Saturday             -213
Good Friday                 -184
Christmas Day               -178
New Year's Day (observed)   -173

Daytime low: mean miss of M3 by period and kind of day (MW)
kind             holiday  weekend  working
period                                    
24 Dec to 2 Jan     -138     -102     -302
rest of year         -84       53      -29
